# Water Fixture Classification — Raw Signal Model

Same pipeline as the feature-based notebook, but the model operates directly on the raw magnetometer time series. Instead of hand-crafted features like `peak_freq_hz` or `ramp_duration_seconds`, an HMM learns its own hidden phase structure from the data.

**How it works:**
- Each fixture type gets its own HMM trained on its labeled signals
- The HMM discovers hidden phases (idle, ramp, plateau, decay, etc.) automatically
- Classification = run every fixture-type HMM on a new signal, pick the one with the lowest free energy (least surprised)
- The transition and emission matrices *are* the learned features — no human defines them

**Sections:**
1. **Labeling Tool** — record fixture events (same as feature-based notebook)
2. **Fetch & Prepare** — pull mag data, discretize into bins
3. **Visualise** — plot raw signals and discretized versions
4. **Train HMMs** — one HMM per fixture type, learned from labeled data
5. **Prediction Tool** — record a new event, classify by free energy comparison

In [1]:
!pip install requests ipywidgets matplotlib numpy pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import math
import threading
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from datetime import datetime, timezone
from IPython.display import display, HTML
import ipywidgets as widgets

matplotlib.rcParams.update({"figure.dpi": 120, "font.size": 10})

In [3]:
# ── Hasura connection ─────────────────────────────────────────

HASURA_URL = "https://hasura.pipestuesday.org/v1/graphql"
HASURA_SECRET = "PIPE_SUPERMMMSECRET_PIPE"
SENSOR_ID = 6

def gql(query, variables=None):
    resp = requests.post(
        HASURA_URL,
        json={"query": query, "variables": variables or {}},
        headers={
            "Content-Type": "application/json",
            "x-hasura-admin-secret": HASURA_SECRET,
        },
    )
    data = resp.json()
    if "errors" in data:
        raise Exception(data["errors"])
    return data["data"]

try:
    gql("query { signal(where: {sensor_id: {_eq: 6}}, limit: 1, order_by: {created_at: desc}) { id } }")
    print("Connected to Hasura")
except Exception as e:
    print(f"Failed to connect: {e}")

Connected to Hasura


In [4]:
# ── GraphQL queries & labeling helpers ─────────────────────────

MUTATION_INSERT = """
mutation InsertSignal(
  $sensor_id: bigint!,
  $start_time: timestamptz!,
  $end_time: timestamptz!,
  $value: String!
) {
  insert_signal_one(object: {
    sensor_id: $sensor_id,
    start_time: $start_time,
    end_time: $end_time,
    value: $value,
    time: $start_time
  }) {
    id
  }
}
"""

QUERY_RECENT = """
query RecentSignals($sensor_id: bigint!) {
  signal(
    where: { sensor_id: { _eq: $sensor_id } }
    order_by: { created_at: desc }
    limit: 5
  ) {
    id
    value
    start_time
    end_time
  }
}
"""

MUTATION_DELETE = """
mutation DeleteSignal($id: bigint!) {
  delete_signal_by_pk(id: $id) {
    id
  }
}
"""

QUERY_SIGNALS = """
query GetSignals {
  signal(
    where: {
      sensor_id: { _eq: 6 }
      value: { _is_null: false }
      start_time: { _is_null: false }
      end_time: { _is_null: false }
    }
    order_by: { start_time: asc }
  ) {
    id
    value
    start_time
    end_time
  }
}
"""

QUERY_MAG_REPORTS = """
query GetMagReports($sensor_id: bigint!, $start: timestamptz!, $end: timestamptz!) {
  mag_report(
    where: {
      sensor_id: { _eq: $sensor_id }
      created_at: { _gte: $start, _lte: $end }
    }
    order_by: { created_at: asc }
  ) {
    id
    created_at
    dominant_freq_hz
    flow_running
    flow_intensity
    total_magnitude
    vibration_rpm
    band_energy_10s
    band_energy_60s
    x_axis_reading
    y_axis_reading
    z_axis_reading
  }
}
"""

def fetch_recent():
    try:
        return gql(QUERY_RECENT, {"sensor_id": SENSOR_ID})["signal"]
    except Exception:
        return []

def save_signal(start_time, end_time, label):
    st = start_time.strftime("%Y-%m-%dT%H:%M:%S.000Z")
    et = end_time.strftime("%Y-%m-%dT%H:%M:%S.000Z")
    data = gql(MUTATION_INSERT, {
        "sensor_id": SENSOR_ID,
        "start_time": st,
        "end_time": et,
        "value": label,
    })
    return data["insert_signal_one"]["id"]

def delete_signal(signal_id):
    gql(MUTATION_DELETE, {"id": signal_id})

In [5]:
# ── Labeling UI (same as feature-based notebook) ──────────────

def fmt_time(iso_str):
    if isinstance(iso_str, datetime):
        dt = iso_str.astimezone()
    else:
        dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00")).astimezone()
    return dt.strftime("%H:%M:%S")

def fmt_duration(seconds):
    seconds = int(seconds)
    if seconds < 60:
        return f"{seconds}s"
    m, s = divmod(seconds, 60)
    return f"{m}m{s:02d}s"

start_time = None
end_time = None
timer_running = False

HEADER_STYLE = "font-size:18px; font-weight:bold; font-family:monospace; padding:8px 0;"
STATUS_STYLE = "font-size:15px; font-family:monospace; padding:4px 0;"
TABLE_STYLE = "font-family:monospace; font-size:13px;"
BTN_LAYOUT = widgets.Layout(width="120px", height="44px")
BTN_LAYOUT_WIDE = widgets.Layout(width="180px", height="44px")

FIXTURE_COLORS = {"toilet": "#e74c3c", "sink": "#3498db", "shower": "#2ecc71", "urinal": "#f39c12"}
def get_color(ftype): return FIXTURE_COLORS.get(ftype, "#999999")

header = widgets.HTML(
    f'<div style="{HEADER_STYLE}">'
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>'
    '&nbsp;Water Fixture Labeler &nbsp;|&nbsp; sensor 6<br>'
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</div>'
)
status_html = widgets.HTML(value="")
recent_html = widgets.HTML(value="")
message_html = widgets.HTML(value="")
btn_start = widgets.Button(description="⏺ Start", button_style="danger", layout=BTN_LAYOUT_WIDE)
btn_stop = widgets.Button(description="⏹ Stop", button_style="warning", layout=BTN_LAYOUT_WIDE)
btn_toilet = widgets.Button(description="🚽 Toilet", button_style="info", layout=BTN_LAYOUT)
btn_sink = widgets.Button(description="🚰 Sink", button_style="info", layout=BTN_LAYOUT)
btn_shower = widgets.Button(description="🚿 Shower", button_style="info", layout=BTN_LAYOUT)
btn_urinal = widgets.Button(description="🚾 Urinal", button_style="info", layout=BTN_LAYOUT)
btn_discard = widgets.Button(description="✕ Discard", button_style="", layout=BTN_LAYOUT)
btn_undo = widgets.Button(description="↩ Undo last", button_style="warning", layout=BTN_LAYOUT_WIDE)
label_buttons = widgets.HBox([btn_toilet, btn_sink, btn_shower, btn_urinal, btn_discard])
idle_buttons = widgets.HBox([btn_start, btn_undo])
recording_buttons = widgets.HBox([btn_stop])
ui = widgets.VBox([header, message_html, status_html, idle_buttons, recording_buttons, label_buttons, recent_html])

def render_recent():
    rows = fetch_recent()
    if not rows:
        recent_html.value = f'<div style="{TABLE_STYLE}; padding-top:12px;">Recent labels: (none)</div>'
        return
    lines = ['<div style="padding-top:12px;">', f'<table style="{TABLE_STYLE}; border-collapse:collapse;">']
    lines.append('<tr style="border-bottom:1px solid #555;"><th style="padding:2px 8px; text-align:left;">ID</th>'
                 '<th style="padding:2px 8px; text-align:left;">Label</th>'
                 '<th style="padding:2px 8px; text-align:left;">Time</th>'
                 '<th style="padding:2px 8px; text-align:left;">Duration</th></tr>')
    for sig in rows:
        if sig["start_time"] is None or sig["end_time"] is None:
            st_str = fmt_time(sig["start_time"]) if sig["start_time"] else "—"
            et_str = fmt_time(sig["end_time"]) if sig["end_time"] else "—"
            dur = "—"
        else:
            st = datetime.fromisoformat(sig["start_time"].replace("Z", "+00:00"))
            et = datetime.fromisoformat(sig["end_time"].replace("Z", "+00:00"))
            dur = fmt_duration((et - st).total_seconds())
            st_str = fmt_time(sig["start_time"])
            et_str = fmt_time(sig["end_time"])
        lines.append(f'<tr><td style="padding:2px 8px;">#{sig["id"]}</td>'
                     f'<td style="padding:2px 8px;">{sig["value"]}</td>'
                     f'<td style="padding:2px 8px;">{st_str} → {et_str}</td>'
                     f'<td style="padding:2px 8px;">{dur}</td></tr>')
    lines.append('</table></div>')
    recent_html.value = '\n'.join(lines)

def show_state_idle(msg=None):
    global start_time, end_time
    start_time = None; end_time = None
    message_html.value = f'<div style="{STATUS_STYLE}; color:#4a4;">{msg}</div>' if msg else ""
    status_html.value = f'<div style="{STATUS_STYLE}">Press <b>Start</b> to begin recording.</div>'
    idle_buttons.layout.display = "flex"
    recording_buttons.layout.display = "none"
    label_buttons.layout.display = "none"
    render_recent()

def show_state_recording():
    message_html.value = ""
    idle_buttons.layout.display = "none"
    recording_buttons.layout.display = "flex"
    label_buttons.layout.display = "none"
    recent_html.value = ""

def show_state_labeling():
    dur = fmt_duration((end_time - start_time).total_seconds())
    status_html.value = (f'<div style="{STATUS_STYLE}">'
        f'Recorded: <b>{fmt_time(start_time)} → {fmt_time(end_time)}</b> ({dur})<br><br>What was it?</div>')
    message_html.value = ""
    idle_buttons.layout.display = "none"
    recording_buttons.layout.display = "none"
    label_buttons.layout.display = "flex"
    recent_html.value = ""

def run_timer():
    global timer_running
    while timer_running:
        elapsed = (datetime.now(timezone.utc) - start_time).total_seconds()
        m, s = divmod(int(elapsed), 60)
        status_html.value = (f'<div style="{STATUS_STYLE}">'
            f'<span style="color:#e44;">⏺ RECORDING</span> <b>{m:02d}:{s:02d}</b><br>'
            f'Started: {fmt_time(start_time)}<br>Go use the fixture, then press <b>Stop</b>.</div>')
        time.sleep(1)

def on_start(_):
    global start_time, timer_running
    start_time = datetime.now(timezone.utc); timer_running = True
    show_state_recording()
    threading.Thread(target=run_timer, daemon=True).start()

def on_stop(_):
    global end_time, timer_running
    timer_running = False; end_time = datetime.now(timezone.utc)
    show_state_labeling()

def make_label_cb(label):
    def cb(_):
        try:
            rid = save_signal(start_time, end_time, label)
            show_state_idle(f"Saved as {label.upper()} (id: {rid})")
        except Exception as e:
            message_html.value = f'<div style="{STATUS_STYLE}; color:#e44;">Save failed: {e}</div>'
    return cb

def on_discard(_): show_state_idle("Discarded.")

def on_undo(_):
    rows = fetch_recent()
    if not rows:
        message_html.value = f'<div style="{STATUS_STYLE}">Nothing to undo.</div>'; return
    last = rows[0]
    try:
        delete_signal(last["id"]); show_state_idle(f"Deleted #{last['id']}")
    except Exception as e:
        message_html.value = f'<div style="{STATUS_STYLE}; color:#e44;">Undo failed: {e}</div>'

btn_start.on_click(on_start); btn_stop.on_click(on_stop)
btn_toilet.on_click(make_label_cb("toilet")); btn_sink.on_click(make_label_cb("sink"))
btn_shower.on_click(make_label_cb("shower")); btn_urinal.on_click(make_label_cb("urinal"))
btn_discard.on_click(on_discard); btn_undo.on_click(on_undo)

show_state_idle()
display(ui)

---

# 2. Fetch & Prepare

Fetch magnetometer data for each labeled event. Instead of extracting hand-crafted features, we keep the raw time series and discretize it into bins for the HMM.

In [ ]:
# ── Fetch labeled signals ─────────────────────────────────────

print("Fetching labeled signals...")
signals = gql(QUERY_SIGNALS)["signal"]
print(f"Found {len(signals)} labeled signal(s)\n")

if not signals:
    raise SystemExit("No labeled signals found. Use the labeling tool above first.")

events = []
event_dfs = {}

for sig in signals:
    sid = sig["id"]
    print(f"  Fetching mag_reports for signal #{sid} ({sig['value']})...")
    readings = gql(QUERY_MAG_REPORTS, {
        "sensor_id": SENSOR_ID,
        "start": sig["start_time"],
        "end": sig["end_time"],
    })["mag_report"]

    if len(readings) < 5:
        print(f"  Warning: Signal #{sid} has only {len(readings)} readings, skipping")
        continue

    df = pd.DataFrame(readings)
    df["created_at"] = pd.to_datetime(df["created_at"], format="ISO8601")
    t0 = df["created_at"].iloc[0]
    df["elapsed_seconds"] = (df["created_at"] - t0).dt.total_seconds()

    # Vibration intensity (prefer 10s, fall back to 60s)
    def get_vib(row):
        v = row.get("band_energy_10s")
        if v is not None: return float(v)
        v = row.get("band_energy_60s")
        if v is not None: return float(v)
        return 0.0
    df["vibration_intensity"] = df.apply(get_vib, axis=1)

    events.append(sig)
    event_dfs[sid] = df

print(f"\n{len(events)} event(s) with sufficient data")

Fetching labeled signals...
Found 31 labeled signal(s)

  Fetching mag_reports for signal #1678 (toilet)...


In [ ]:
# ── Discretization ────────────────────────────────────────────
#
# Instead of raw x/y/z amplitudes (which have identical distributions
# for toilet and sink), we use FIRST DIFFERENCES (dx, dy, dz).
#
# dx[t] = x[t] - x[t-1]  — how fast the magnetometer reading changed
#
# Fast oscillation (toilet) → large positive/negative jumps
# Slow oscillation (sink) → small gradual changes
#
# This directly encodes what you see with your eyes: the oscillation
# speed and pattern. We also include the MAGNITUDE of the 3D change
# vector as a 4th channel.
#
# Channels:
#   dx (x-axis change)  → 3 bins: negative, near-zero, positive
#   dy (y-axis change)  → 3 bins: negative, near-zero, positive
#   dz (z-axis change)  → 3 bins: negative, near-zero, positive
#   |d| (change magnitude) → 3 bins: small, medium, large
#
# Combined: 3^4 = 81 observation symbols

N_BINS_PER_CHANNEL = 3
N_CHANNELS = 4
N_OBS = N_BINS_PER_CHANNEL ** N_CHANNELS  # 81
N_STATES = 5

# ── Step 1: compute global bin edges from ALL events ──────────

all_dx, all_dy, all_dz, all_dmag = [], [], [], []

for sig in events:
    df = event_dfs[sig["id"]]
    x = df["x_axis_reading"].astype(float).values
    y = df["y_axis_reading"].astype(float).values
    z = df["z_axis_reading"].astype(float).values
    dx = np.diff(x)
    dy = np.diff(y)
    dz = np.diff(z)
    dmag = np.sqrt(dx**2 + dy**2 + dz**2)
    all_dx.extend(dx)
    all_dy.extend(dy)
    all_dz.extend(dz)
    all_dmag.extend(dmag)

all_dx = np.array(all_dx)
all_dy = np.array(all_dy)
all_dz = np.array(all_dz)
all_dmag = np.array(all_dmag)

# For directional channels: negative / near-zero / positive
# Use 33rd and 67th percentiles
DX_EDGES = np.quantile(all_dx, [1/3, 2/3])
DY_EDGES = np.quantile(all_dy, [1/3, 2/3])
DZ_EDGES = np.quantile(all_dz, [1/3, 2/3])
# For magnitude: small / medium / large
DMAG_EDGES = np.quantile(all_dmag, [1/3, 2/3])

print(f"Global derivative bin edges:")
print(f"  dX: {DX_EDGES}  (neg/zero/pos)")
print(f"  dY: {DY_EDGES}  (neg/zero/pos)")
print(f"  dZ: {DZ_EDGES}  (neg/zero/pos)")
print(f"  |d|: {DMAG_EDGES}  (small/med/large)")

# ── Step 2: discretize ────────────────────────────────────────

def bin_channel(values, edges):
    return np.clip(np.digitize(values, edges), 0, N_BINS_PER_CHANNEL - 1)

def discretize_signal_xyz(df):
    """Discretize first-differences of x/y/z mag readings."""
    x = df["x_axis_reading"].astype(float).values
    y = df["y_axis_reading"].astype(float).values
    z = df["z_axis_reading"].astype(float).values

    dx = np.diff(x)
    dy = np.diff(y)
    dz = np.diff(z)
    dmag = np.sqrt(dx**2 + dy**2 + dz**2)

    dx_bins = bin_channel(dx, DX_EDGES)
    dy_bins = bin_channel(dy, DY_EDGES)
    dz_bins = bin_channel(dz, DZ_EDGES)
    dmag_bins = bin_channel(dmag, DMAG_EDGES)

    obs = (dx_bins * N_BINS_PER_CHANNEL**3 +
           dy_bins * N_BINS_PER_CHANNEL**2 +
           dz_bins * N_BINS_PER_CHANNEL +
           dmag_bins)

    # Store AC for plotting
    x_ac = x - x.mean()
    y_ac = y - y.mean()
    z_ac = z - z.mean()

    return obs, dx_bins, dy_bins, dz_bins, x_ac[1:], y_ac[1:], z_ac[1:]

def obs_to_label(obs_idx):
    names = ["neg", "zero", "pos"]
    mag_names = ["small", "med", "large"]
    dx = obs_idx // (N_BINS_PER_CHANNEL**3)
    remainder = obs_idx % (N_BINS_PER_CHANNEL**3)
    dy = remainder // (N_BINS_PER_CHANNEL**2)
    remainder2 = remainder % (N_BINS_PER_CHANNEL**2)
    dz = remainder2 // N_BINS_PER_CHANNEL
    dm = remainder2 % N_BINS_PER_CHANNEL
    return f"dx{names[dx]}_dy{names[dy]}_dz{names[dz]}_{mag_names[dm]}"

# Discretize all events
event_obs = {}
for sig in events:
    sid = sig["id"]
    df = event_dfs[sid]
    obs, dxb, dyb, dzb, x_ac, y_ac, z_ac = discretize_signal_xyz(df)
    event_obs[sid] = obs
    # Store on truncated df (diff loses one row)
    df_trunc = df.iloc[1:].copy()
    df_trunc["dx_bin"] = dxb
    df_trunc["dy_bin"] = dyb
    df_trunc["dz_bin"] = dzb
    df_trunc["x_ac"] = x_ac
    df_trunc["y_ac"] = y_ac
    df_trunc["z_ac"] = z_ac
    df_trunc["obs"] = obs
    event_dfs[sid] = df_trunc

    unique_obs = np.unique(obs)
    print(f"Signal #{sid} ({sig['value']}): {len(obs)} readings, "
          f"{len(unique_obs)} unique symbols out of {N_OBS}")

---

# 3. Visualise

Plot the raw signals, discretized observation sequences, and the joint freq-vibration heatmaps.

In [ ]:
# ── Plot 1: Raw signals with vibration overlay ────────────────

fixture_types = sorted(set(s["value"] for s in events))
n_types = max(len(fixture_types), 1)

fig, axes = plt.subplots(2, n_types, figsize=(6 * n_types, 6), squeeze=False,
                         gridspec_kw={"height_ratios": [2, 1]})

for idx, ftype in enumerate(fixture_types):
    ax_freq = axes[0, idx]
    ax_vib = axes[1, idx]
    color = get_color(ftype)
    type_events = [s for s in events if s["value"] == ftype]
    n = len(type_events)

    for sig in type_events:
        df = event_dfs[sig["id"]]
        ax_freq.plot(df["elapsed_seconds"], df["dominant_freq_hz"].astype(float),
                     color=color, alpha=0.5, linewidth=1)
        ax_vib.plot(df["elapsed_seconds"], df["vibration_intensity"],
                    color=color, alpha=0.5, linewidth=1)

    ax_freq.set_title(f"{ftype} (n={n})")
    ax_freq.set_ylabel("dominant_freq_hz")
    ax_vib.axhline(5.0, color="red", linestyle="--", alpha=0.4, label="noise threshold")
    ax_vib.set_ylabel("vibration_intensity")
    ax_vib.set_xlabel("elapsed seconds")
    ax_vib.legend(fontsize=8)

fig.suptitle("Raw Signals — frequency (top) & vibration (bottom)", fontweight="bold")
fig.tight_layout()
os.makedirs("plots_raw", exist_ok=True)
fig.savefig("plots_raw/raw_signals.png")
plt.show()

In [ ]:
# ── Plot 2: Discretized derivative sequences ──────────────────

fig, axes = plt.subplots(3, n_types, figsize=(10 * n_types, 6), squeeze=False)
bin_names = ["dx_bin", "dy_bin", "dz_bin"]
bin_labels = ["dx (neg/zero/pos)", "dy (neg/zero/pos)", "dz (neg/zero/pos)"]

for idx, ftype in enumerate(fixture_types):
    type_events = [s for s in events if s["value"] == ftype]
    color = get_color(ftype)

    for row, (bname, blabel) in enumerate(zip(bin_names, bin_labels)):
        ax = axes[row, idx]
        for sig in type_events:
            df = event_dfs[sig["id"]]
            ax.step(df["elapsed_seconds"], df[bname], color=color, alpha=0.4,
                    where="post", linewidth=1)
        ax.set_ylabel(blabel)
        ax.set_yticks(range(N_BINS_PER_CHANNEL))
        ax.set_yticklabels(["neg", "~zero", "pos"], fontsize=8)
        if row == 0:
            ax.set_title(f"{ftype} — derivative bins")
        if row == 2:
            ax.set_xlabel("elapsed seconds")

fig.suptitle("Discretized First-Differences — HMM input\n"
             "(fast oscillation = rapid neg↔pos switching)", fontweight="bold")
fig.tight_layout()
fig.savefig("plots_raw/discretized_signals.png")
plt.show()

In [ ]:
# ── Plot 3: Derivative transition patterns ────────────────────

fig, axes = plt.subplots(n_types, 3, figsize=(12, 4 * n_types), squeeze=False)
bin_cols = ["dx_bin", "dy_bin", "dz_bin"]
bin_labels = ["dx", "dy", "dz"]

for row_idx, ftype in enumerate(fixture_types):
    type_events = [s for s in events if s["value"] == ftype]
    color = get_color(ftype)

    for col_idx, (bcol, blabel) in enumerate(zip(bin_cols, bin_labels)):
        ax = axes[row_idx, col_idx]
        all_bins = []
        for sig in type_events:
            df = event_dfs[sig["id"]]
            all_bins.extend(df[bcol].values)
        all_bins = np.array(all_bins)

        counts = np.bincount(all_bins.astype(int), minlength=N_BINS_PER_CHANNEL)
        counts = counts / counts.sum() if counts.sum() > 0 else counts
        bars = ax.bar(range(N_BINS_PER_CHANNEL), counts, color=color, alpha=0.7, edgecolor="white")
        ax.set_xticks(range(N_BINS_PER_CHANNEL))
        ax.set_xticklabels(["neg", "~zero", "pos"], fontsize=9)
        ax.set_ylabel("fraction")
        if row_idx == 0:
            ax.set_title(f"{blabel} distribution")
        if col_idx == 0:
            ax.set_ylabel(f"{ftype}\nfraction")

fig.suptitle("Derivative Bin Distributions\n"
             "(toilet should have more neg/pos, less zero = faster oscillation)", fontweight="bold")
fig.tight_layout()
fig.savefig("plots_raw/observation_heatmaps.png")
plt.show()

---

# 4. Train HMMs

Train one HMM per fixture type on the discretized observation sequences. Each HMM learns:
- **Transition matrix A** (N_STATES x N_STATES) — how the hidden phases flow into each other
- **Emission matrix B** (N_STATES x N_OBS) — what freq+vibration pattern each phase produces

The transition and emission matrices *are* the learned features. The model discovers structure like "phase 2 always emits medium-freq + active-vibration" without anyone telling it to look for a plateau.

**With n=1**, the model will overfit to that single example. That's fine — it gives us a starting point, and additional labeled events will refine the matrices.

In [ ]:
# ── Python HMM (Baum-Welch EM) ───────────────────────────────
#
# Mirrors the RxInfer discrete HMM but runs in Python so the
# entire notebook is self-contained. The RxInfer version
# (fixture_model_raw.jl) uses message passing for exact same result.

class DiscreteHMM:
    """Discrete HMM with Baum-Welch training and free energy scoring."""

    def __init__(self, n_states, n_obs, name=""):
        self.n_states = n_states
        self.n_obs = n_obs
        self.name = name

        # Initialise with slight randomness to break symmetry
        rng = np.random.RandomState(42)
        self.pi = np.ones(n_states) / n_states
        self.A = np.ones((n_states, n_states)) / n_states + rng.dirichlet(np.ones(n_states), n_states) * 0.1
        self.A /= self.A.sum(axis=1, keepdims=True)
        self.B = np.ones((n_states, n_obs)) / n_obs + rng.dirichlet(np.ones(n_obs), n_states) * 0.1
        self.B /= self.B.sum(axis=1, keepdims=True)

    def forward(self, obs):
        """Forward algorithm. Returns alpha matrix and log-likelihood."""
        T = len(obs)
        K = self.n_states
        alpha = np.zeros((T, K))
        scales = np.zeros(T)

        alpha[0] = self.pi * self.B[:, obs[0]]
        scales[0] = alpha[0].sum()
        if scales[0] > 0:
            alpha[0] /= scales[0]

        for t in range(1, T):
            alpha[t] = (alpha[t-1] @ self.A) * self.B[:, obs[t]]
            scales[t] = alpha[t].sum()
            if scales[t] > 0:
                alpha[t] /= scales[t]

        log_likelihood = np.sum(np.log(scales + 1e-300))
        return alpha, scales, log_likelihood

    def backward(self, obs, scales):
        """Backward algorithm."""
        T = len(obs)
        K = self.n_states
        beta = np.zeros((T, K))
        beta[T-1] = 1.0

        for t in range(T-2, -1, -1):
            beta[t] = self.A @ (self.B[:, obs[t+1]] * beta[t+1])
            if scales[t+1] > 0:
                beta[t] /= scales[t+1]

        return beta

    def fit(self, sequences, n_iter=20, verbose=True):
        """Baum-Welch EM training on one or more observation sequences."""
        for iteration in range(n_iter):
            # Accumulators
            A_num = np.zeros_like(self.A)
            A_den = np.zeros(self.n_states)
            B_num = np.zeros_like(self.B)
            B_den = np.zeros(self.n_states)
            pi_acc = np.zeros(self.n_states)
            total_ll = 0.0

            for obs in sequences:
                T = len(obs)
                alpha, scales, ll = self.forward(obs)
                beta = self.backward(obs, scales)
                total_ll += ll

                # Gamma (state posteriors)
                gamma = alpha * beta
                gamma_sum = gamma.sum(axis=1, keepdims=True)
                gamma_sum[gamma_sum == 0] = 1.0
                gamma /= gamma_sum

                # Xi (transition posteriors)
                for t in range(T - 1):
                    xi = np.outer(alpha[t], self.B[:, obs[t+1]] * beta[t+1]) * self.A
                    xi_sum = xi.sum()
                    if xi_sum > 0:
                        xi /= xi_sum
                    A_num += xi
                    A_den += gamma[t]

                for t in range(T):
                    B_num[:, obs[t]] += gamma[t]
                    B_den += gamma[t]

                pi_acc += gamma[0]

            # M-step with smoothing
            smooth = 1e-6
            for i in range(self.n_states):
                if A_den[i] > 0:
                    self.A[i] = (A_num[i] + smooth) / (A_den[i] + smooth * self.n_states)
                if B_den[i] > 0:
                    self.B[i] = (B_num[i] + smooth) / (B_den[i] + smooth * self.n_obs)

            self.pi = (pi_acc + smooth) / (pi_acc.sum() + smooth * self.n_states)

            if verbose and (iteration == 0 or iteration == n_iter - 1):
                print(f"  iter {iteration+1:3d}: log-likelihood = {total_ll:.2f}")

        return total_ll

    def score(self, obs):
        """Score a sequence. Returns log-likelihood (higher = better fit)
        and free energy (lower = better fit, comparable to RxInfer)."""
        _, _, ll = self.forward(obs)
        free_energy = -ll  # negative log-likelihood as free energy proxy
        return ll, free_energy

    def decode(self, obs):
        """Viterbi decoding — MAP state sequence."""
        T = len(obs)
        K = self.n_states
        delta = np.zeros((T, K))
        psi = np.zeros((T, K), dtype=int)

        delta[0] = np.log(self.pi + 1e-300) + np.log(self.B[:, obs[0]] + 1e-300)

        for t in range(1, T):
            for j in range(K):
                scores = delta[t-1] + np.log(self.A[:, j] + 1e-300)
                psi[t, j] = np.argmax(scores)
                delta[t, j] = scores[psi[t, j]] + np.log(self.B[j, obs[t]] + 1e-300)

        # Backtrack
        path = np.zeros(T, dtype=int)
        path[T-1] = np.argmax(delta[T-1])
        for t in range(T-2, -1, -1):
            path[t] = psi[t+1, path[t+1]]

        return path

    def to_dict(self):
        return {
            "name": self.name,
            "n_states": self.n_states,
            "n_obs": self.n_obs,
            "pi": self.pi.tolist(),
            "A": self.A.tolist(),
            "B": self.B.tolist(),
        }

    @classmethod
    def from_dict(cls, d):
        hmm = cls(d["n_states"], d["n_obs"], name=d.get("name", ""))
        hmm.pi = np.array(d["pi"])
        hmm.A = np.array(d["A"])
        hmm.B = np.array(d["B"])
        return hmm

print(f"HMM class ready: {N_STATES} hidden states, {N_OBS} observation symbols")

In [ ]:
# ── Train one HMM per fixture type ────────────────────────────

trained_hmms = {}

for ftype in fixture_types:
    type_events = [s for s in events if s["value"] == ftype]
    sequences = [event_obs[s["id"]] for s in type_events]
    n = len(sequences)

    print(f"\nTraining HMM for '{ftype}' ({n} sequence(s), "
          f"lengths: {[len(s) for s in sequences]})")

    hmm = DiscreteHMM(N_STATES, N_OBS, name=ftype)
    hmm.fit(sequences, n_iter=30)

    # Score training data
    for sig in type_events:
        ll, fe = hmm.score(event_obs[sig["id"]])
        print(f"  Signal #{sig['id']}: log-lik={ll:.2f}, free_energy={fe:.2f}")

    trained_hmms[ftype] = hmm

# Save trained models
model_data = {name: hmm.to_dict() for name, hmm in trained_hmms.items()}
model_data["_meta"] = {
    "n_states": N_STATES, "n_obs": N_OBS,
    "n_bins_per_channel": N_BINS_PER_CHANNEL, "n_channels": N_CHANNELS,
    "dx_edges": DX_EDGES.tolist(), "dy_edges": DY_EDGES.tolist(),
    "dz_edges": DZ_EDGES.tolist(), "dmag_edges": DMAG_EDGES.tolist(),
}
with open("trained_hmms.json", "w") as f:
    json.dump(model_data, f, indent=2)
print(f"\nSaved trained_hmms.json ({len(trained_hmms)} model(s))")

In [ ]:
# ── Visualise learned HMM structure ───────────────────────────

for ftype, hmm in trained_hmms.items():
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    color = get_color(ftype)

    # Transition matrix
    ax = axes[0]
    im = ax.imshow(hmm.A, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"{ftype} — transition matrix A")
    ax.set_xlabel("to state")
    ax.set_ylabel("from state")
    for i in range(N_STATES):
        for j in range(N_STATES):
            ax.text(j, i, f"{hmm.A[i,j]:.2f}", ha="center", va="center", fontsize=7)
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Emission matrix (reshaped as freq x vib per state)
    ax = axes[1]
    # Show emission as heatmap: states x observation symbols
    im = ax.imshow(hmm.B, aspect="auto", cmap="YlOrRd")
    ax.set_title(f"{ftype} — emission matrix B")
    ax.set_xlabel("observation symbol")
    ax.set_ylabel("hidden state")
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Decoded state sequence for each training signal
    ax = axes[2]
    type_events_list = [s for s in events if s["value"] == ftype]
    for sig in type_events_list:
        df = event_dfs[sig["id"]]
        obs = event_obs[sig["id"]]
        path = hmm.decode(obs)
        ax.step(df["elapsed_seconds"], path, color=color, alpha=0.7,
                where="post", linewidth=1.5, label=f"#{sig['id']}")
    ax.set_title(f"{ftype} — decoded hidden states")
    ax.set_xlabel("elapsed seconds")
    ax.set_ylabel("hidden state")
    ax.set_yticks(range(N_STATES))
    ax.legend(fontsize=8)

    fig.tight_layout()
    fig.savefig(f"plots_raw/hmm_{ftype}.png")
    plt.show()

## Test Model

Score every labeled event against all trained HMMs. The model with the lowest free energy wins. Since we're testing on training data, accuracy should be high — low accuracy means the HMMs aren't learning distinct patterns yet.

In [ ]:
# ── Test: classify every labeled event against all HMMs ─────

test_results = []

for sig in events:
    sid = sig["id"]
    true_label = sig["value"]
    obs = event_obs[sid]

    scores = {}
    for ft, hmm in trained_hmms.items():
        _, fe = hmm.score(obs)
        scores[ft] = fe

    predicted = min(scores, key=scores.get)

    # Confidence via softmax over negative free energies
    fes = np.array([scores[ft] for ft in sorted(scores.keys())])
    neg_fes = -fes
    neg_fes -= neg_fes.max()
    exp_fes = np.exp(neg_fes)
    probs = exp_fes / exp_fes.sum()
    prob_dict = dict(zip(sorted(scores.keys()), probs))
    conf = prob_dict[predicted]

    correct = predicted == true_label
    test_results.append({
        "signal_id": sid,
        "true_label": true_label,
        "predicted": predicted,
        "confidence": conf,
        "correct": correct,
        "scores": scores,
    })

n_correct = sum(r["correct"] for r in test_results)
n_total = len(test_results)
acc = (n_correct / n_total * 100) if n_total > 0 else 0

# Build results table
html = ['<div style="font-family:monospace; font-size:14px;">']
html.append('<b>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</b><br>')
html.append('<b>&nbsp;Raw Signal HMM — Test on Training Data</b><br>')
html.append('<b>━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</b><br><br>')

# Header
html.append('<table style="font-family:monospace; font-size:13px; border-collapse:collapse;">')
header = ('<tr style="border-bottom:1px solid #555;">'
          '<th style="padding:4px 8px;">Signal</th>'
          '<th style="padding:4px 8px;">True</th>')
for ft in sorted(trained_hmms.keys()):
    header += f'<th style="padding:4px 8px;">FE({ft})</th>'
header += ('<th style="padding:4px 8px;">Predicted</th>'
           '<th style="padding:4px 8px;">Conf</th>'
           '<th style="padding:4px 8px;">Correct?</th></tr>')
html.append(header)

for r in test_results:
    check = '<span style="color:#2ecc71;">yes</span>' if r["correct"] else '<span style="color:#e74c3c;">NO</span>'
    row = (f'<td style="padding:4px 8px;">#{r["signal_id"]}</td>'
           f'<td style="padding:4px 8px;">{r["true_label"]}</td>')
    for ft in sorted(trained_hmms.keys()):
        fe = r["scores"][ft]
        style = 'color:#2ecc71; font-weight:bold;' if ft == r["predicted"] else ''
        row += f'<td style="padding:4px 8px; {style}">{fe:.1f}</td>'
    row += (f'<td style="padding:4px 8px; font-weight:bold;">{r["predicted"]}</td>'
            f'<td style="padding:4px 8px;">{r["confidence"]:.0%}</td>'
            f'<td style="padding:4px 8px;">{check}</td>')
    html.append(f'<tr>{row}</tr>')

html.append('</table><br>')
html.append(f'<b>Accuracy: {n_correct}/{n_total} ({acc:.0f}%)</b><br>')

if n_total <= 2:
    html.append('<i style="color:#ea0;">Very few events — accuracy is not meaningful yet. '
                'Label more events and re-run.</i><br>')

# Per-type breakdown
if n_total > 0:
    html.append('<br><b>Per fixture type:</b><br>')
    for ftype in sorted(set(r["true_label"] for r in test_results)):
        subset = [r for r in test_results if r["true_label"] == ftype]
        nc = sum(r["correct"] for r in subset)
        nt = len(subset)
        a = (nc / nt * 100) if nt > 0 else 0
        html.append(f'&nbsp;&nbsp;{ftype}: {nc}/{nt} ({a:.0f}%)<br>')

html.append('</div>')
display(HTML('\n'.join(html)))

---

# 5. Prediction Tool

Record a new fixture event (start/stop), fetch its magnetometer data, discretize it, run it through every trained HMM, and classify by lowest free energy.

The model tells you:
- **Which fixture type** the signal most resembles
- **Free energy per model** — how surprised each HMM was
- **Decoded state sequence** — what hidden phases the model thinks the signal went through

In [ ]:
# ── Prediction UI ─────────────────────────────────────────────

pred_start_time = None
pred_end_time = None
pred_timer_running = False

pred_header = widgets.HTML(
    '<div style="font-size:18px; font-weight:bold; font-family:monospace; padding:8px 0;">'
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━<br>'
    '&nbsp;Fixture Predictor (Raw Signal) &nbsp;|&nbsp; sensor 6<br>'
    '━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━</div>'
)

pred_status = widgets.HTML(value="")
pred_result = widgets.HTML(value="")
pred_plot_out = widgets.Output()

pred_btn_start = widgets.Button(description="⏺ Start", button_style="danger",
                                 layout=widgets.Layout(width="180px", height="44px"))
pred_btn_stop = widgets.Button(description="⏹ Stop & Predict", button_style="warning",
                                layout=widgets.Layout(width="180px", height="44px"))

pred_start_box = widgets.HBox([pred_btn_start])
pred_stop_box = widgets.HBox([pred_btn_stop])

pred_ui = widgets.VBox([
    pred_header, pred_status,
    pred_start_box, pred_stop_box,
    pred_result, pred_plot_out,
])

def pred_show_idle():
    pred_status.value = ('<div style="font-size:14px; font-family:monospace; padding:4px 0;">'
                         'Press <b>Start</b>, go use a fixture, come back and press '
                         '<b>Stop & Predict</b>.</div>')
    pred_start_box.layout.display = "flex"
    pred_stop_box.layout.display = "none"

def pred_show_recording():
    pred_start_box.layout.display = "none"
    pred_stop_box.layout.display = "flex"
    pred_result.value = ""
    pred_plot_out.clear_output()

def pred_run_timer():
    global pred_timer_running
    while pred_timer_running:
        elapsed = (datetime.now(timezone.utc) - pred_start_time).total_seconds()
        m, s = divmod(int(elapsed), 60)
        pred_status.value = (
            '<div style="font-size:14px; font-family:monospace; padding:4px 0;">'
            f'<span style="color:#e44;">⏺ RECORDING</span> <b>{m:02d}:{s:02d}</b><br>'
            f'Started: {fmt_time(pred_start_time)}<br>'
            'Go use the fixture, then press <b>Stop & Predict</b>.</div>'
        )
        time.sleep(1)

def on_pred_start(_):
    global pred_start_time, pred_timer_running
    pred_start_time = datetime.now(timezone.utc)
    pred_timer_running = True
    pred_show_recording()
    threading.Thread(target=pred_run_timer, daemon=True).start()

def on_pred_stop(_):
    global pred_end_time, pred_timer_running
    pred_timer_running = False
    pred_end_time = datetime.now(timezone.utc)
    dur = (pred_end_time - pred_start_time).total_seconds()
    pred_status.value = (
        '<div style="font-size:14px; font-family:monospace; padding:4px 0;">'
        f'Recorded: {fmt_time(pred_start_time)} → {fmt_time(pred_end_time)} ({fmt_duration(dur)})<br>'
        'Fetching data and classifying...</div>'
    )
    pred_start_box.layout.display = "none"
    pred_stop_box.layout.display = "none"
    threading.Thread(target=run_prediction, daemon=True).start()

def run_prediction():
    try:
        # Fetch mag data
        st = pred_start_time.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        et = pred_end_time.strftime("%Y-%m-%dT%H:%M:%S.000Z")
        readings = gql(QUERY_MAG_REPORTS, {
            "sensor_id": SENSOR_ID, "start": st, "end": et,
        })["mag_report"]

        if len(readings) < 5:
            pred_result.value = (
                '<div style="font-size:14px; font-family:monospace; padding:8px; '
                'color:#e44; border:1px solid #e44; border-radius:6px; margin:8px 0;">'
                f'Only {len(readings)} reading(s). Need at least 5. Try a longer recording.</div>'
            )
            pred_show_idle()
            return

        # Prepare DataFrame
        df = pd.DataFrame(readings)
        df["created_at"] = pd.to_datetime(df["created_at"], format="ISO8601")
        t0 = df["created_at"].iloc[0]
        df["elapsed_seconds"] = (df["created_at"] - t0).dt.total_seconds()

        # Discretize first-differences of x/y/z
        obs, dxb, dyb, dzb, x_ac, y_ac, z_ac = discretize_signal_xyz(df)

        # Score against all trained HMMs
        scores = {}
        for ft, hmm in trained_hmms.items():
            ll, fe = hmm.score(obs)
            scores[ft] = {"ll": ll, "fe": fe}

        predicted = min(scores, key=lambda ft: scores[ft]["fe"])

        # Confidence: softmax over negative free energies
        fes = np.array([scores[ft]["fe"] for ft in sorted(scores.keys())])
        neg_fes = -fes
        neg_fes -= neg_fes.max()
        exp_fes = np.exp(neg_fes)
        probs = exp_fes / exp_fes.sum()
        prob_dict = dict(zip(sorted(scores.keys()), probs))
        conf = prob_dict[predicted]

        if conf > 0.7:
            conf_color = "#2ecc71"; conf_label = "high"
        elif conf > 0.4:
            conf_color = "#f39c12"; conf_label = "medium"
        else:
            conf_color = "#e74c3c"; conf_label = "low"

        # Result HTML
        result_html = (
            '<div style="font-size:16px; font-family:monospace; padding:12px; '
            f'border:2px solid {conf_color}; border-radius:8px; margin:8px 0;">'
            f'<b>Prediction: <span style="color:{conf_color};">{predicted.upper()}</span></b>'
            f' ({conf:.0%} confidence, {conf_label})<br><br>'
        )

        for ft in sorted(scores.keys()):
            fe = scores[ft]["fe"]
            p = prob_dict[ft]
            bar_width = int(p * 200)
            color = FIXTURE_COLORS.get(ft, "#999")
            bold = "font-weight:bold;" if ft == predicted else ""
            result_html += (
                f'<div style="margin:2px 0; {bold}">'
                f'<span style="display:inline-block; width:80px;">{ft}</span>'
                f'<span style="display:inline-block; width:{bar_width}px; height:16px; '
                f'background:{color}; border-radius:3px; margin-right:8px;"></span>'
                f'{p:.1%} (FE={fe:.1f})</div>'
            )

        result_html += f'<br><i>{len(obs)} readings, {len(np.unique(obs))}/{N_OBS} unique symbols</i>'
        result_html += '</div>'
        pred_result.value = result_html

        # Plot: raw x/y/z AC + decoded states
        best_hmm = trained_hmms[predicted]
        path = best_hmm.decode(obs)
        elapsed = df["elapsed_seconds"].values[1:]  # diff loses one row

        with pred_plot_out:
            pred_plot_out.clear_output(wait=True)
            fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)

            x_raw = df["x_axis_reading"].astype(float).values
            y_raw = df["y_axis_reading"].astype(float).values
            z_raw = df["z_axis_reading"].astype(float).values

            axes[0].plot(df["elapsed_seconds"], x_raw - x_raw.mean(),
                         color="#e74c3c", alpha=0.7, linewidth=0.8, label="x")
            axes[0].plot(df["elapsed_seconds"], y_raw - y_raw.mean(),
                         color="#2ecc71", alpha=0.7, linewidth=0.8, label="y")
            axes[0].plot(df["elapsed_seconds"], z_raw - z_raw.mean(),
                         color="#3498db", alpha=0.7, linewidth=0.8, label="z")
            axes[0].set_ylabel("mag AC")
            axes[0].set_title(f"Prediction: {predicted.upper()} — raw x/y/z & decoded states")
            axes[0].legend(fontsize=8)
            axes[0].axhline(0, color="gray", linestyle="--", alpha=0.3)

            ac_mag = np.sqrt((x_raw - x_raw.mean())**2 +
                            (y_raw - y_raw.mean())**2 +
                            (z_raw - z_raw.mean())**2)
            axes[1].plot(df["elapsed_seconds"], ac_mag, color=conf_color, linewidth=1)
            axes[1].set_ylabel("AC magnitude")

            axes[2].step(elapsed, path, color=conf_color,
                         where="post", linewidth=1.5)
            axes[2].set_ylabel("hidden state")
            axes[2].set_xlabel("elapsed seconds")
            axes[2].set_yticks(range(N_STATES))

            fig.tight_layout()
            plt.show()

    except Exception as e:
        pred_result.value = (
            '<div style="font-size:14px; font-family:monospace; padding:8px; '
            f'color:#e44; border:1px solid #e44; border-radius:6px;">Error: {e}</div>'
        )

    pred_show_idle()

pred_btn_start.on_click(on_pred_start)
pred_btn_stop.on_click(on_pred_stop)

pred_show_idle()
display(pred_ui)

---

# 6. RxInfer Model (Julia)

The Python HMM above mirrors what RxInfer does with message passing. To use the Julia version:

```julia
include("Water Fixture Classification/fixture_model_raw.jl")

# Train on discretized sequences
train_and_save("events/", "trained_hmms_rxinfer.json")

# Classify a new observation sequence
predicted, free_energies = classify_raw(obs_sequence, "trained_hmms_rxinfer.json")
```

The RxInfer version uses variational message passing (Bethe free energy) rather than Baum-Welch EM,
which gives a proper lower bound on the model evidence — useful for detecting out-of-distribution signals.

---

# Reference

## Feature-based vs Raw Signal model

| | Feature-based | Raw Signal (this notebook) |
|---|---|---|
| **Input** | 9 hand-crafted numbers | Full time series (~50ms resolution) |
| **Features** | Human-defined (peak_freq, ramp_duration, etc.) | Learned by the HMM (transition + emission matrices) |
| **What it can catch** | Only patterns you thought to encode | Any temporal pattern in freq + vibration |
| **Data needed** | Works OK with n=1 | Works with n=1 but overfits; improves significantly with n=5+ |
| **Classification** | Gaussian log-likelihood per feature | Free energy comparison across per-type HMMs |
| **Interpretability** | Easy — you can read the feature values | Moderate — need to inspect decoded state sequences |
| **Speed** | Instant | ~1 second for training, instant for classification |

## How the HMM discovers features

The HMM doesn't extract features like `ramp_duration_seconds`. Instead, it learns:
- **State 0** might become "idle" — emits `none+silent` symbols, stays in state 0
- **State 2** might become "plateau" — emits `medium+active` symbols, self-loops
- **State 4** might become "decay" — emits `low+low` symbols, transitions to state 0

Nobody told the model these phases exist. It discovered them because they explain the training data well.
The transition matrix captures ramp duration implicitly (how long it stays in state 1 before reaching state 2).
The emission matrix captures vibration gating implicitly (state 2 emits `medium+active`, not `medium+silent`).

## Discretization bins

| Channel | Bins | Rationale |
|---------|------|----------|
| **Frequency** | silent (<0.003), trace (0.003-0.008), low (0.008-0.015), medium (0.015-0.025), high (>0.025) | Tuned to actual sensor range (plateau freq ~0.004-0.025 Hz) |
| **Vibration** | noise (<5), low (5-10), moderate (10-14), strong (>14) | Separates noise floor from real signal; splits sink (~12) vs toilet (~10.5) |

Combined into 20 symbols so the HMM can learn joint patterns like "high frequency only when vibration is strong".

## Free energy for classification

Each fixture type has its own HMM. To classify a new signal:
1. Run the forward algorithm on the new signal with each HMM
2. Each returns a free energy (negative log-likelihood)
3. The HMM with the **lowest** free energy is the least surprised — that's the prediction

This is fundamentally different from the feature-based model. Instead of asking "do these 9 numbers look like a toilet?",
it asks "does this entire 800-reading time series look like something a toilet-trained model would generate?"

## When to retrain

Re-run the training cells after:
- Labeling new events (more data = better models)
- Adding a new fixture type
- Changing the discretization bins (if the default ranges don't fit your sensor)

The trained models are saved to `trained_hmms.json` — the prediction tool loads from there.